# Optimisation complète — toutes familles, toutes optimisations

Pipeline unique appliquant **en séquence** chaque optimisation (et combinaisons
raisonnables) sur **les 6 modèles**, avec robustesse, fallback et journalisation.

**Modèles** : RetinaNet R50, RetinaNet R101*, FCOS R50, EfficientDet D4/D5/D6
\*R101 = backbone ImageNet + tête aléatoire → vitesse uniquement (MAP ~0).

**Variantes par modèle**

| Variante | Ce qu'elle fait | MAP évaluée ? |
|---|---|:---:|
| `baseline` | FP32 brut | ✓ (référence) |
| `fp16` | autocast demi-précision (Tensor Cores) | ✓ (impact précision) |
| `torchscript` | graphe + fusion Conv-BN | ✗ (≈ baseline FP32) |
| `compile` | torch.compile inductor (`dynamic=False`) | ✗ (≈ baseline FP32) |
| `compile_fp16` | autocast + compile (combinaison) | ✗ |
| `trt_fp16` | TensorRT FP16 (fusion + précision) | ✓ |
| `trt_int8` | TensorRT INT8 + calibration (optionnel) | ✓ |

**Robustesse** : chaque (modèle, variante) est isolé. Un échec est journalisé
(`errors/`) et n'interrompt pas le reste. `results.csv` est réécrit après chaque
variante → survit à une déconnexion Colab.

**MAP à 640×640** : la MAP réutilise le modèle optimisé (entrée fixe 640) — c'est
le **delta** baseline→optimisé qui mesure l'impact FP16/INT8, pas la valeur absolue.

## 0. Installation (Colab)

```bash
!pip install torch-tensorrt onnx onnxruntime-gpu onnxsim effdet pycocotools --quiet
```
TensorRT/CUDA sont pré-installés dans le runtime Colab GPU. `effdet` est requis
pour EfficientDet. **Plus aucune dépendance Detectron2** (R101 est en torchvision).

In [ ]:
# Colab — décommenter à la première exécution
# !pip install torch-tensorrt onnx onnxruntime-gpu onnxsim effdet pycocotools --quiet

# Monter Google Drive pour sauvegarder les résultats de façon persistante (optionnel)
# from google.colab import drive; drive.mount('/content/drive')

## 1. Imports, capacités & configuration

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))   # racine du projet (adapter si besoin)

from datetime import datetime
from pathlib import Path

import torch
import pandas as pd

import optimizations as opt
from optimizations import OptimizationRunner, RunConfig, ModelSpec
from optimizations.runner import DEFAULT_VARIANTS
from utils.data_loader import load_profiling_data, load_eval_data

# ── Capacités de l'environnement ──────────────────────────────────────────────
CAPS = opt.print_report()

# ── Paramètres expérience ─────────────────────────────────────────────────────
N_WARMUP   = 50
N_MEASURE  = 1000
N_PROFILE  = 2000          # images chargées pour le benchmark (>= warmup+measure)
N_EVAL     = 2000          # images pour la MAP@640
DO_INT8    = False         # passer à True pour activer TRT INT8 (calibration ~5 min)
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

IMG_DIR  = "datasets/coco/val2017"
ANN_FILE = "datasets/coco/annotations/instances_val2017.json"

# ── Répertoire de run horodaté (résultats + logs + modèles) ───────────────────
RUN_ID  = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path("results/optimization") / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
Path("outputs/models").mkdir(parents=True, exist_ok=True)
Path("outputs/onnx").mkdir(parents=True, exist_ok=True)

print(f"\nDevice  : {DEVICE}")
print(f"Run dir : {RUN_DIR}")
if DEVICE == "cpu":
    print("⚠ Aucun GPU CUDA — le benchmark nécessite CUDA. Vérifier torch (+cuXXX).")

## 2. Données (profilage + évaluation)

In [ ]:
from pycocotools.coco import COCO

profile_data    = load_profiling_data(IMG_DIR, ANN_FILE, n=N_PROFILE)
eval_data, _    = load_eval_data(IMG_DIR, ANN_FILE, n=N_EVAL)   # unpack (LazyList, coco)
coco_gt         = COCO(ANN_FILE)

assert len(profile_data) >= N_WARMUP + N_MEASURE, \
    f"profile_data={len(profile_data)} < warmup+measure={N_WARMUP+N_MEASURE}"

print(f"Profilage : {len(profile_data)} images")
print(f"Eval MAP  : {len(eval_data)} images")

## 3. Déclaration des modèles et des variantes

`has_map=False` pour R101 (tête aléatoire). Les EfficientDet utilisent un jeu de
variantes **sans `compile`** par défaut : `torch.compile`/inductor sur le BiFPN est
pathologiquement lent (solveur de shapes). Mettre `EFFDET_WITH_COMPILE=True` pour l'activer.

In [ ]:
import models.retinanet_r50  as r50
import models.retinanet_r101 as r101
import models.fcos_r50       as fcos
import models.efficientdet_d4 as d4
import models.efficientdet_d5 as d5
import models.efficientdet_d6 as d6

SPECS = {
    "retinanet_r50":   ModelSpec("retinanet_r50",   r50,  "torchvision", has_map=True),
    "retinanet_r101":  ModelSpec("retinanet_r101",  r101, "torchvision", has_map=False),
    "fcos_r50":        ModelSpec("fcos_r50",        fcos, "torchvision", has_map=True),
    "efficientdet_d4": ModelSpec("efficientdet_d4", d4,   "effdet",      has_map=True),
    "efficientdet_d5": ModelSpec("efficientdet_d5", d5,   "effdet",      has_map=True),
    "efficientdet_d6": ModelSpec("efficientdet_d6", d6,   "effdet",      has_map=True),
}

# Variantes torchvision = jeu complet
VARIANTS_TV = DEFAULT_VARIANTS

# Variantes effdet = sans compile (risque de compilation très lente sur BiFPN)
EFFDET_WITH_COMPILE = False
VARIANTS_EFFDET = [v for v in DEFAULT_VARIANTS
                   if EFFDET_WITH_COMPILE or v.name not in ("compile", "compile_fp16")]

print("Variantes torchvision :", [v.name for v in VARIANTS_TV])
print("Variantes effdet      :", [v.name for v in VARIANTS_EFFDET])

## 4. Construction de l'orchestrateur

In [ ]:
config = RunConfig(
    n_warmup        = N_WARMUP,
    n_measure       = N_MEASURE,
    device          = DEVICE,
    compile_backend = CAPS.compile_backend,   # "inductor" sur Colab
    trt_available   = CAPS.flags["tensorrt"],
    do_int8         = DO_INT8,
    int8_calib_images = 300,
    min_block_size  = 5,
)

runner = OptimizationRunner(
    profile_data, eval_data, coco_gt,
    config=config, run_dir=str(RUN_DIR),
)
print("Orchestrateur prêt. Logs →", RUN_DIR / "run.log")

## 5. Exécution — un modèle par cellule

Chaque cellule est indépendante : si l'une plante/déconnecte, les résultats déjà
écrits dans `results.csv` sont conservés. Relancer une cellule réexécute juste ce modèle.

In [ ]:
runner.run_model(SPECS["retinanet_r50"], VARIANTS_TV)

In [ ]:
runner.run_model(SPECS["retinanet_r101"], VARIANTS_TV)

In [ ]:
runner.run_model(SPECS["fcos_r50"], VARIANTS_TV)

In [ ]:
runner.run_model(SPECS["efficientdet_d4"], VARIANTS_EFFDET)

In [ ]:
runner.run_model(SPECS["efficientdet_d5"], VARIANTS_EFFDET)

In [ ]:
runner.run_model(SPECS["efficientdet_d6"], VARIANTS_EFFDET)

## 6. Résultats agrégés

In [ ]:
df = runner.to_dataframe()
df.to_csv(RUN_DIR / "results_final.csv", index=False)
print("Statuts :", dict(df["status"].value_counts()))
df

In [ ]:
# Tableau speedup (robuste aux modèles sans baseline)
spd = runner.speedup_table()
spd.to_csv(RUN_DIR / "speedup.csv", index=False)
spd

In [ ]:
# Échecs éventuels — résumé + pointeur vers les tracebacks complets
fails = df[df["status"] == "FAILED"][["model", "variant", "error"]]
if len(fails):
    print("Échecs (traceback complet dans", RUN_DIR / "errors", ") :")
    display(fails)
else:
    print("Aucun échec.")

## 7. Analyse par module — d'où viennent les gains

Décomposition Conv / BatchNorm / activation du baseline de chaque modèle
(les briques que TorchScript replie et que TensorRT fusionne en kernel CBR).

In [ ]:
mod_dir = RUN_DIR / "modules"
rows = []
for csv_path in sorted(mod_dir.glob("*_baseline.csv")):
    model = csv_path.stem.replace("_baseline", "")
    dfm = pd.read_csv(csv_path)
    total = dfm["mean_ms"].sum()
    def part(pat):
        return dfm[dfm["type"].str.contains(pat, na=False, regex=True)]["mean_ms"].sum()
    conv, bn, act = part("Conv"), part("BatchNorm|FrozenBatch"), part("ReLU|SiLU|Swish")
    rows.append({
        "model": model, "total_ms": round(total, 2),
        "Conv_%":  round(100*conv/total, 1) if total else 0,
        "BN_%":    round(100*bn/total, 1) if total else 0,
        "Act_%":   round(100*act/total, 1) if total else 0,
        "CBR_fusible_%": round(100*(conv+bn+act)/total, 1) if total else 0,
    })
df_modbreak = pd.DataFrame(rows)
df_modbreak.to_csv(RUN_DIR / "module_breakdown.csv", index=False)
print("Part du temps fusionnable (Conv+BN+activation) par modèle :")
df_modbreak

## 8. Vue transversale — opérations les plus accélérables

(Point 2 du cahier des charges) Agrège les modules de **tous** les modèles par
type d'opération, pour identifier les briques élémentaires qui dominent le temps
et répondent le mieux à la fusion / FP16 / INT8.

In [ ]:
frames = []
for csv_path in sorted(mod_dir.glob("*_baseline.csv")):
    model = csv_path.stem.replace("_baseline", "")
    dfm = pd.read_csv(csv_path); dfm["model"] = model
    frames.append(dfm)

if frames:
    allmod = pd.concat(frames, ignore_index=True)
    by_type = (allmod.groupby("type")
               .agg(total_ms=("mean_ms", "sum"),
                    mean_ms=("mean_ms", "mean"),
                    n=("mean_ms", "size"))
               .sort_values("total_ms", ascending=False)
               .round(3))
    by_type.to_csv(RUN_DIR / "ops_transversal.csv")
    print("Opérations par temps cumulé (tous modèles confondus) :")
    display(by_type.head(20))
else:
    print("Aucun CSV de modules — exécuter les modèles d'abord.")

## 9. Sauvegarde finale

Tous les artefacts sont dans le répertoire de run :

```
results/optimization/<RUN_ID>/
  run.log                 ← journal complet (console + fichier)
  results.csv             ← incrémental (après chaque variante)
  results_final.csv       ← copie finale
  speedup.csv             ← speedup par variante
  module_breakdown.csv    ← décomposition Conv/BN/act par modèle
  ops_transversal.csv     ← vue transversale des opérations
  modules/<model>_<variant>.csv
  errors/<model>_<variant>.txt
```

Pour conserver après la session Colab, copier le dossier sur Drive :
```python
import shutil; shutil.copytree(RUN_DIR, f"/content/drive/MyDrive/{RUN_ID}")
```

In [ ]:
# Copier les résultats sur Google Drive (si monté)
# import shutil
# dest = f"/content/drive/MyDrive/optimization_{RUN_ID}"
# shutil.copytree(RUN_DIR, dest, dirs_exist_ok=True)
# print("Copié →", dest)
print("Run terminé. Artefacts dans :", RUN_DIR)